## Notebook 3 - Feature Engineering

In [1]:
#Imports
import pandas as pd
import numpy as np

In [2]:
# 1. LOAD DATA AND MERGE SQL FEATURES

df     = pd.read_csv("cleaned_rossmann.csv", parse_dates=['Date'])
sql_df = pd.read_csv("sql_features.csv",     parse_dates=['Date'])

df = df.merge(sql_df, on=['Store', 'Date'], how='left')
print("After merge shape:", df.shape)

After merge shape: (844338, 29)


In [3]:
# 2. SORT DATA

df = df.sort_values(['Store', 'Date']).reset_index(drop=True)


In [4]:
# 3. LAG FEATURES

lags = [7, 14, 28]
for lag in lags:
    df[f'lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)

# lag_365: date-based merge on (Store, Date - 1 year) — correct calendar alignment
df_lag365 = df[['Store', 'Date', 'Sales']].copy()
df_lag365['Date'] = df_lag365['Date'] + pd.DateOffset(years=1)
df_lag365 = df_lag365.rename(columns={'Sales': 'lag_365'})
df = df.merge(df_lag365, on=['Store', 'Date'], how='left')

# Fill lag nulls with store rolling mean (avoids 48% data loss)
for lag in lags + [365]:
    col = f'lag_{lag}'
    df[col] = df[col].fillna(
        df.groupby('Store')['Sales'].transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
    )

lag_cols = [f'lag_{l}' for l in lags + [365]]
print("Lag nulls after fill:", df[lag_cols].isnull().sum().sum())
print("Shape after lag features:", df.shape)



Lag nulls after fill: 4460
Shape after lag features: (844338, 33)


Note:
4460 nulls remain due to lag features (e.g., 365-day lag) where early dates lack history.
These repeat across rows since each store appears multiple times (~1115 stores).

In [5]:
# DEBUG LAG NULLS

# Identify rows where lag/rolling features are still missing (likely first-year data after lag_365 merge)
# Helps verify whether nulls are expected (no history) or due to incorrect feature construction

lag_nulls = df[df.isnull().any(axis=1)]

print("Lag null rows sample:")
print(lag_nulls[['Store','Date']].head(10))

print("Total lag null rows:", len(lag_nulls))

#Remove Lag Nulls
df = df.dropna(subset=lag_cols).reset_index(drop=True)

Lag null rows sample:
      Store       Date
0         1 2013-01-02
781       2 2013-01-02
1565      3 2013-01-02
2344      4 2013-01-02
3128      5 2013-01-02
3907      6 2013-01-02
4687      7 2013-01-02
5473      8 2013-01-02
6257      9 2013-01-02
7036     10 2013-01-02
Total lag null rows: 1115


In [6]:
# 4. COMPETITION FEATURES
# CompetitionDistance already log-transformed in NB1 (log1p), so fillna(0) is safe (log1p(0)=0)

df['CompetitionDistance'] = df['CompetitionDistance'].fillna(0)

df['CompetitionOpen'] = np.where(
    df['CompetitionOpenSinceYear'] > 0,
    (df['Year']  - df['CompetitionOpenSinceYear'])  * 12 +
    (df['Month'] - df['CompetitionOpenSinceMonth']),
    0    # 0 = no competition / unknown open date
)
df['CompetitionOpen'] = df['CompetitionOpen'].clip(lower=0)


In [7]:
# 5. PROMO FEATURES

month_map = {
    'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5,  'Jun':6,
    'Jul':7, 'Aug':8, 'Sept':9,'Oct':10,'Nov':11, 'Dec':12
}

def extract_months(x):
    if x == 'none' or pd.isna(x):
        return []
    try:
        return [month_map[m.strip()] for m in x.split(',')]
    except KeyError:
        return []   # Unknown abbreviation — treat as no promo month

df['PromoMonths'] = df['PromoInterval'].apply(extract_months)
df['IsPromoMonth'] = df.apply(lambda row: int(row['Month'] in row['PromoMonths']), axis=1)
df.drop(columns=['PromoMonths'], inplace=True)


In [8]:
# 6. ENCODING CATEGORICALS

STORETYPE_MAP   = {'a': 0, 'b': 1, 'c': 2, 'd': 3}
ASSORTMENT_MAP  = {'a': 0, 'b': 1, 'c': 2}
HOLIDAY_MAP     = {'none': 0, 'public': 1, 'easter': 2, 'christmas': 3}

df['StoreType']    = df['StoreType'].map(STORETYPE_MAP).fillna(-1).astype(int)
df['Assortment']   = df['Assortment'].map(ASSORTMENT_MAP).fillna(-1).astype(int)
df['StateHoliday'] = df['StateHoliday'].map(HOLIDAY_MAP).fillna(0).astype(int)

# State: sort alphabetically for determinism then encode
state_categories = sorted(df['State'].dropna().unique())
STATE_MAP = {s: i for i, s in enumerate(state_categories)}
df['State'] = df['State'].map(STATE_MAP).fillna(-1).astype(int)

print("StoreType codes :", STORETYPE_MAP)
print("State codes (first 5):", dict(list(STATE_MAP.items())[:5]))


StoreType codes : {'a': 0, 'b': 1, 'c': 2, 'd': 3}
State codes (first 5): {'BB': 0, 'BE': 1, 'BW': 2, 'BY': 3, 'HB,NI': 4}


In [9]:
# 7. FEATURE LIST

FEATURE_COLS = [
    'Store', 'DayOfWeek', 'Promo', 'SchoolHoliday',
    'Year', 'Month', 'Week', 'IsWeekend',
    'StoreType', 'Assortment', 'StateHoliday', 'State',
    'CompetitionDistance', 'CompetitionOpen',
    'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'IsPromoMonth',
    'lag_7', 'lag_14', 'lag_28', 'lag_365',
    'avg_4w', 'std_4w', 'median_4w', 'avg_1w', 'dev_4w'
]


In [10]:
# 8. VALIDATION

missing_cols = [c for c in FEATURE_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

null_counts = df[FEATURE_COLS].isnull().sum()
if null_counts.sum() > 0:
    print("WARNING — nulls remain:")
    print(null_counts[null_counts > 0])
else:
    print("All feature columns clean — no nulls")

print("Final shape:", df.shape)
df.head()


All feature columns clean — no nulls
Final shape: (843223, 35)


,Store,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,Year,Month,...,std_4w,median_4w,avg_1w,dev_4w,lag_7,lag_14,lag_28,lag_365,CompetitionOpen,IsPromoMonth
0,1,2013-01-03,4327,578,1,0,0,1,2013,1,...,850.649458,5530.0,5530.0,-1203.0,5530.0,5530.0,5530.0,5530.0,52.0,0
1,1,2013-01-04,4486,619,1,0,0,1,2013,1,...,850.649458,4928.5,4928.5,-442.5,4928.5,4928.5,4928.5,4928.5,52.0,0
2,1,2013-01-05,4997,635,1,0,0,1,2013,1,...,653.506695,4486.0,4781.0,216.0,4781.0,4781.0,4781.0,4781.0,52.0,0
3,1,2013-01-07,7176,785,1,1,0,1,2013,1,...,544.406098,4741.5,4835.0,2341.0,4835.0,4835.0,4835.0,4835.0,52.0,0
4,1,2013-01-08,5580,654,1,1,0,1,2013,1,...,1148.189749,4997.0,5303.2,276.8,5303.2,5303.2,5303.2,5303.2,52.0,0


In [11]:
# 9. SAVE

df.to_csv("rossmann_features.csv", index=False)

with open("feature_cols.txt", "w") as f:
    f.write("\n".join(FEATURE_COLS))

print("Saved: rossmann_features.csv")
print("Saved: feature_cols.txt")


Saved: rossmann_features.csv
Saved: feature_cols.txt
